# HuggingFace Pipelines — Hands-On Exploration

> **Environment:** Google Colab (GPU: Tesla T4)  
> **Libraries:** `transformers`, `diffusers`, `datasets`

---

## Overview

This notebook explores the **HuggingFace `pipeline` API** — a high-level interface that lets you run state-of-the-art NLP, vision, and audio models with just a few lines of code.

The general pattern is:

```python
# Step 1: Create the pipeline (loads model + tokenizer)
my_pipeline = pipeline("task-name", model="optional/model-id", device="cuda")

# Step 2: Run inference — as many times as you want
result = my_pipeline("Your input text here")
```

### Tasks Covered

| # | Task | Pipeline Type |
|---|------|--------------|
| 1 | Sentiment Analysis | `sentiment-analysis` |
| 2 | Named Entity Recognition | `ner` |
| 3 | Question Answering | `question-answering` |
| 4 | Text Summarization | `summarization` |
| 5 | Translation (EN→FR, EN→ES) | `translation_en_to_*` |
| 6 | Zero-Shot Classification | `zero-shot-classification` |
| 7 | Text Generation | `text-generation` |
| 8 | Image Generation | `AutoPipelineForText2Image` (Diffusers) |
| 9 | Text-to-Speech | `text-to-speech` |

---

### Training vs. Inference

- **Training** — The model learns by adjusting its weights on labelled data. *Fine-tuning* is training a model that has already been pre-trained.
- **Inference** — Using a trained model to produce predictions on new inputs.

The `pipeline` API is **inference-only**. All models used here are pre-trained and ready to use out of the box.

---
## Setup

### 1. Install Dependencies

In [ ]:
# Pin versions for reproducibility
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

### 2. Verify GPU

These experiments run on a **Tesla T4** GPU. If you see `NOT CONNECTED TO A T4`, go to **Runtime → Change runtime type → GPU**.

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)
    if gpu_info.find('Tesla T4') >= 0:
        print('Connected to a T4')
    else:
        print('NOT CONNECTED TO A T4')

### 3. Imports

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio, display

### 4. HuggingFace Authentication

A free HuggingFace account is required to access gated models. Create one at [huggingface.co](https://huggingface.co), generate a token with **write** permissions under *Settings → Access Tokens*, then add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith('hf_'):
    print('HF token found')
else:
    print('HF_TOKEN not set — add it via the key icon in the left sidebar')

login(hf_token, add_to_git_credential=True)

---
## Experiments

### 1. Sentiment Analysis

Classify whether text carries a **positive** or **negative** sentiment.

- **Default model** (`distilbert-base-uncased-finetuned-sst-2-english`): binary positive/negative.
- **Multilingual model** (`nlptown/bert-base-multilingual-uncased-sentiment`): 1–5 star rating, supports multiple languages.

In [ ]:
# Default binary sentiment
sentiment = pipeline('sentiment-analysis', device='cuda')

samples = [
    "I'm super excited to be on the way to LLM mastery!",
    "I should be more excited to be on the way to LLM mastery!",
]

for text in samples:
    result = sentiment(text)
    label, score = result[0]['label'], result[0]['score']
    print(f"[{label} | {score:.2%}]  {text}")

In [ ]:
# Multilingual 5-star sentiment — more nuanced than binary
detailed_sentiment = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    device='cuda'
)
result = detailed_sentiment('I should be more excited to be on the way to LLM mastery!!')
print(result)

### 2. Named Entity Recognition (NER)

Identify and classify **named entities** in text — people, organisations, locations, and more.

In [ ]:
ner = pipeline('ner', device='cuda')

text = 'AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner'
entities = ner(text)

print(f'Input: {text}\n')
print('Detected entities:')
for entity in entities:
    print(f"  {entity['word']:<20} -> {entity['entity']}  (confidence: {entity['score']:.2%})")

### 3. Question Answering (Extractive)

Given a **context passage** and a **question**, the model extracts the answer span directly from the context. This is *extractive* QA — the answer must appear verbatim in the context.

In [ ]:
qa = pipeline('question-answering', device='cuda')

question = 'What are Hugging Face pipelines?'
context  = 'Pipelines are a high level API for inference of LLMs with common tasks'

result = qa(question=question, context=context)
print(f"Question : {question}")
print(f"Answer   : {result['answer']}  (score: {result['score']:.2%})")

### 4. Text Summarization

Condense a longer document into a short summary. `max_length` and `min_length` control output length in tokens.

In [ ]:
summarizer = pipeline('summarization', device='cuda')

text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print('Summary:', summary[0]['summary_text'])

### 5. Translation

Translate text between languages. Browse hundreds of available models at [huggingface.co/models?pipeline_tag=translation](https://huggingface.co/models?pipeline_tag=translation&sort=trending).

In [ ]:
# English -> French (default model)
translator_fr = pipeline('translation_en_to_fr', device='cuda')
text = 'The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.'

result = translator_fr(text)
print('EN:', text)
print('FR:', result[0]['translation_text'])

In [ ]:
# English -> Spanish with an explicit Helsinki-NLP model
# Specifying model= gives more control over quality and language pair
translator_es = pipeline(
    'translation_en_to_es',
    model='Helsinki-NLP/opus-mt-en-es',
    device='cuda'
)
result = translator_es(text)
print('EN:', text)
print('ES:', result[0]['translation_text'])

### 6. Zero-Shot Classification

Classify text into **arbitrary categories without any task-specific training**. Supply the candidate labels at inference time — no fine-tuning required.

In [ ]:
classifier = pipeline('zero-shot-classification', device='cuda')

text   = "Hugging Face's Transformers library is amazing!"
labels = ['technology', 'sports', 'politics']

result = classifier(text, candidate_labels=labels)

print(f'Input: {text}\n')
for label, score in zip(result['labels'], result['scores']):
    bar = '#' * int(score * 30)
    print(f'  {label:<12} {score:.2%}  {bar}')

### 7. Text Generation

Auto-regressively complete a text prompt. The default model (`gpt2`) is small and fast; swap in a larger model for higher-quality outputs.

In [ ]:
generator = pipeline('text-generation', device='cuda')

prompt = "If there's one thing I want you to remember about using HuggingFace pipelines, it's"
result = generator(prompt, max_new_tokens=60, do_sample=True, temperature=0.8)

print(result[0]['generated_text'])

### 8. Image Generation (Diffusion)

Generate images from text prompts using **Stable Diffusion XL Turbo** — a distilled diffusion model that produces results in as few as 4 inference steps.

> Note: `AutoPipelineForText2Image` comes from the `diffusers` library (not `transformers`), but follows the same conceptual pattern.

In [ ]:
from diffusers import AutoPipelineForText2Image

# Load SDXL-Turbo in fp16 to fit within T4 VRAM
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=torch.float16,
    variant='fp16'
)
pipe.to('cuda')

prompt = 'A class of students learning AI engineering in a vibrant pop-art style'

# guidance_scale=0.0 is required for SDXL-Turbo (classifier-free guidance is disabled)
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

### 9. Text-to-Speech

Synthesise natural-sounding speech from text using Microsoft's **SpeechT5** model. Speaker identity is controlled via an **x-vector embedding** drawn from the CMU-ARCTIC dataset.

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import torch
from IPython.display import Audio

# Load TTS pipeline
synthesiser = pipeline('text-to-speech', 'microsoft/speecht5_tts', device='cuda')

# Load speaker embeddings (x-vectors encode a speaker's vocal characteristics)
embeddings_dataset = load_dataset('matthijs/cmu-arctic-xvectors', split='validation', trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]['xvector']).unsqueeze(0)

speech = synthesiser(
    'Hi to an artificial intelligence engineer, on the way to mastery!',
    forward_params={'speaker_embeddings': speaker_embedding}
)

Audio(speech['audio'], rate=speech['sampling_rate'])

---
## Summary & Key Takeaways

The HuggingFace `pipeline` API demonstrates how much the open-source ML ecosystem has democratised AI:

- **Two lines to inference** — load a model and call it. No manual tokenization or post-processing.
- **Task-first design** — specify *what* you want, not *how* to do it. Swap in a custom `model=` when defaults aren't enough.
- **Unified interface across modalities** — the same pattern works for text, images, and audio.
- **Production-ready** — pin model and library versions (as in the install cell) for reproducible results.

### Further Reading

- [Transformers pipeline docs](https://huggingface.co/docs/transformers/main_classes/pipelines) — full list of supported tasks
- [Diffusers pipeline docs](https://huggingface.co/docs/diffusers/en/api/pipelines/overview) — image, video, and audio generation
- [Model Hub](https://huggingface.co/models) — browse 500,000+ pre-trained models
